## Het feit rit S2

In [1]:
#SETUP
import ConnectionConfig as cc
cc.setupEnvironment()
from pyspark.sql import SparkSession

In [2]:
#SETUP
spark = cc.startLocalCluster("factRides")
spark.getActiveSession()


In [3]:
#EXTRACT
cc.set_connectionProfile("velodb")

ride_src_df = spark.read \
    .format("jdbc") \
    .option("url", cc.create_jdbc()) \
    .option("driver", cc.get_Property("driver")) \
    .option("dbtable", "rides") \
    .option("user", cc.get_Property("username")) \
    .option("password", cc.get_Property("password")) \
    .option("partitionColumn", "rideId") \
    .option("numPartitions", 4) \
    .option("lowerBound", 0) \
    .option("upperBound", 1000) \
    .load()

In [4]:
#EXTRACT
dim_customer = spark.read.format("delta").load("spark-warehouse/dimcustomer")
dim_locks = spark.read.format("delta").load("spark-warehouse/dimlocks")

In [5]:
#TRANSFORM
dim_customer.createOrReplaceTempView("dimCustomer")
dim_locks.createOrReplaceTempView("dimLocks")

In [6]:
#TRANSFORM
from pyspark.sql.functions import udf
from pyspark.sql.types import FloatType
import math


def haversine_km(lat1, lon1, lat2, lon2):
    if lat1 == lat2 and lon1 == lon2:
        return 0.0
    else:
        lat1_rad = math.radians(lat1)
        lat2_rad = math.radians(lat2)
        delta_lon = math.radians(lon2 - lon1)
        distance = 6371 * math.acos(
            math.cos(lat1_rad) * math.cos(lat2_rad) * math.cos(delta_lon) + math.sin(lat1_rad) * math.sin(lat2_rad))
        return distance


# Register UDF
haversine_km_udf = udf(haversine_km, FloatType())
spark.udf.register("haversine_km", haversine_km_udf)

# todo DOES NOT WORK, WHY?
# spark.sql("""
#     CREATE OR REPLACE FUNCTION haversine_km(lat1 numeric, lon1 numeric, lat2 numeric, lon2 numeric)
#     RETURNS numeric AS $$
#     DECLARE
#     distance numeric;
#     BEGIN
#     IF lat1 = lat2 AND lon1 = lon2 THEN
#           RETURN 0;
#     ELSE
#           SELECT 6371 * ACOS(
#              COS(radians(lat1)) 
#             * COS(radians(lat2)) 
#             * COS(radians(lon2) - radians(lon1)) 
#             + SIN(radians(lat1)) * SIN(radians(lat2))
#         ) INTO distance;
#         RETURN distance;
#     END IF;
#     END;
#     $$ LANGUAGE plpgsql IMMUTABLE;
# """)

In [7]:
#TRANSFORM
ride_src_df.createOrReplaceTempView("rides_source")

In [8]:
#TRANSFORM
fact_rides = spark.sql("""
    SELECT 
        r.rideId AS rideId,
        s1.lockId AS startLockId,
        s2.lockId AS endLockId,
        s1.lockSK AS startLockSK,
        s2.lockSK AS endLockSK,
        r.starttime AS starttime,
        r.endtime AS endtime,
        c.userId AS userId,
        haversine_km(
            CAST(SUBSTRING(r.startpoint FROM 2 FOR POSITION(',' IN r.startpoint)-2) AS FLOAT), 
            CAST(SUBSTRING(r.startpoint FROM POSITION(',' IN r.startpoint)+1 FOR LENGTH(r.startpoint)-POSITION(',' IN r.startpoint)-1) AS FLOAT),
            CAST(SUBSTRING(r.endpoint FROM 2 FOR POSITION(',' IN r.endpoint)-2) AS FLOAT), 
            CAST(SUBSTRING(r.endpoint FROM POSITION(',' IN r.endpoint)+1 FOR LENGTH(r.endpoint)-POSITION(',' IN r.endpoint)-1) AS FLOAT)
        ) AS distance
    FROM
        rides_source AS r
    LEFT OUTER JOIN 
        dimLocks AS s1 ON r.startlockId = s1.lockId
    LEFT OUTER JOIN 
        dimLocks AS s2 ON r.endlockId = s2.lockId
    LEFT OUTER JOIN 
        dimCustomer AS c ON r.subscriptionid = c.subscriptionid
    WHERE 
        s1.lockId IS NOT NULL AND s2.lockId IS NOT NULL AND c.userId IS NOT NULL
        and r.starttime >= c.scd_start and (r.starttime < c.scd_end  OR c.scd_end is null)
""")

In [9]:
#LOAD
fact_rides.show(20)
fact_rides.write.format("delta").mode("overwrite").saveAsTable("factRides")

+------+-----------+---------+-----------+---------+-------------------+-------------------+------+----------+
|rideId|startLockId|endLockId|startLockSK|endLockSK|          starttime|            endtime|userId|  distance|
+------+-----------+---------+-----------+---------+-------------------+-------------------+------+----------+
|     1|       4849|     3188|        992|      681|2015-09-22 00:00:00|2012-09-22 00:00:00| 10231| 3.4434342|
|     3|       2046|     1951|       4147|     2270|2015-09-22 00:00:00|2012-09-22 00:00:00| 19786|0.69910353|
|     5|       6382|     2700|       5171|      589|2015-09-22 00:00:00|2012-09-22 00:00:00| 45784| 4.8841357|
|     8|       2572|       13|       5991|       12|2015-09-22 00:00:00|2012-09-22 00:00:00| 49947| 1.4019372|
|    11|        985|     2148|       5628|     4177|2015-09-22 00:00:00|2012-09-22 00:00:00|   755| 1.0614833|
|    12|       2039|     3038|       4140|     6169|2015-09-22 00:00:00|2012-09-22 00:00:00| 45872| 2.5453248|
|

In [10]:
#CLEANUP
spark.stop()